# Out-Degree

### Imports

In [1]:
from __future__ import annotations
import networkx as nx
import numpy as np
from abc import ABC, abstractmethod
import pytest

### Error Classes

In [ ]:
class NormalizationError(Exception):
    """Exception raised for errors in the normalization.

    Attributes:
        message -- explanation of the error
    """

    def __init__(self, message="An error occurred during normalization."):
        self.message = message
        super().__init__(self.message)

class NullGraphError(Exception):
    """Exception raised for null graph."""
    pass


def check_raw_value(func):
    """Decorator to check if raw value is None. If it is, raise an error."""
    def wrapper(self, *args):
        if self._raw_value is not None:
            return func(self, *args)
        else:
            raise ValueError("Raw value is None. Call compute() method first.")
    return wrapper

### Abstract Class

In [2]:
class _Property(ABC):
    """Abstract base class for all properties."""
    _return_type = None
    _use_paths = False
    _use_direction = False
    _use_selfloops = False
    _use_giant_component = False

    def __init__(self, G: nx.DiGraph):
        self.G = G
        self._raw_value = None
        self._n_nodes = self.G.number_of_nodes()
        if self._n_nodes == 0:
            raise NullGraphError("A null graph has no out-degree distribution.")

    @abstractmethod
    def compute(self):
        return self._raw_value

    @abstractmethod
    def norm_biol(self, *args):
        pass

    @abstractmethod
    def norm_network(self, *args):
        pass

### Decorators

In [3]:
def use_direction(cls):
    cls._use_direction = True
    return cls

def use_selfloops(cls):
    cls._use_selfloops = True
    return cls

def use_giant_component(cls):
    cls._use_giant_component = True
    return cls

def return_scalar(cls):
    cls._return_type = "scalar"
    return cls

def return_distribution(cls):
    cls._return_type = "distribution"
    return cls

def use_paths(cls):
    cls._use_paths = True
    return cls

### Out-Degree Class
Degree connectivity for a node is the number of edges it forms with other nodes, or itself in the case of a network with self-loops. For directed networks the connectivity is divided between an out-degree and an in-degree.
Out degree of a node is the number of successors it has.

In [7]:
@return_distribution
@use_direction
@use_selfloops  
class Out_Degree(_Property): # Hereda de la clase _Property
    """Out degree of the graph.

    The out degree of each node is defined as the number of successors it has.

    Methods:
        compute: Compute the out connectivity of the graph.
        norm_biol: Normalize the out connectivity of the graph to the number of nodes.
        norm_network: Normalize the out connectivity of the graph to the number of nodes.
    """
    __name__ = 'Out-Degree'

    def __init__(self, G: nx.DiGraph):
        """
        Args:
            G (nx.DiGraph): Graph.
        """
        super().__init__(G)

    def compute(self) -> np.array:
        """Compute the out-degree of the graph.

        Returns:
            nparray: Array with out-degrees of every node in graph.    
        """
        self._raw_value = np.array([x for a,x in self.G.out_degree()])
        return self._raw_value

    @check_raw_value    # Decorator to check if raw value is None. If it is, raise an error.
    def norm_biol(self) -> float:
        """Normalize the nparray with the out-degrees of the graph to the number of nodes"""
        return self._raw_value * (1 / self._n_nodes)

    @check_raw_value
    def norm_network(self) -> float:
        """Normalize the nparray with the out-degrees of the graph to the number of nodes."""
        return self._raw_value * (1 / self._n_nodes)  

### Testing

In [6]:
# null graph
G = nx.DiGraph()
with pytest.raises(NullGraphError) as e_info:
    property = Out_Degree(G)

# empty graph with nodes
G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))    # add nodes, no edges

# no edges
property = Out_Degree(G)
assert property.compute().all() == 0 # no edges [0, 0, ..., 0]
assert property.norm_biol().all() == 0 # no edges [0, 0, ..., 0] * 1 / 10
assert property.norm_network().all() == 0 # no edges [0, 0, ..., 0] * 1/10

# add edges
# complete directed graph with self loops
expected = np.array([10,10,10,10,10,10,10,10,10,10])
expected_1 = np.array([1.,1.,1.,1.,1.,1.,1.,1.,1.,1.])

G.add_edges_from([(i, j) for i in range(n_nodes) for j in range(n_nodes)])
property = Out_Degree(G)
assert np.array_equal(property.compute(), expected) # [10, 10, ..., 10]
assert np.array_equal(property.norm_network(), expected_1) #[10, 10, ..., 10] * (1 / 10) = [1., 1., ..., 1.]
assert np.array_equal(property.norm_biol(), expected_1) #[10, 10, ..., 10] * (1 / 10) = [1., 1., ..., 1.]

# only half of the nodes are parents and regulate every node in the graph
expected = np.array([10,10,10,10,10,0,0,0,0,0])
expected_1 = np.array([1.,1.,1.,1.,1.,0,0,0,0,0])
G = nx.DiGraph()
n_nodes = 10
G.add_nodes_from(range(n_nodes))
G.add_edges_from([(i, j) for i in range(n_nodes//2) for j in range(n_nodes)])
property = Out_Degree(G)
assert np.array_equal(property.compute(), expected) # [10, 10, 10, 10, 10, 0, 0, 0, 0, 0]
assert np.array_equal(property.norm_network(), expected_1) #[10, 10, 10, 10, 10, 0, 0, 0, 0, 0] * (1 / 10) = [1., 1., 1., 1., 1., 0, 0, 0, 0, 0]
assert np.array_equal(property.norm_biol(), expected_1) #[10, 10, 10, 10, 10, 0, 0, 0, 0, 0] * (1 / 10) = [1., 1., 1., 1., 1., 0, 0, 0, 0, 0]
